# ASCON-AEAD-128 Dataset Verification

Checks chunk counts, trace/metadata alignment, and metadata columns.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

PROJECT_DIR = Path(r"C:\Users\thetp\Downloads\ASCON_FYP")
DATASET_ROOT = PROJECT_DIR / r"03_Data\tight_trigger_keybyte_auto30k"

# Change this after collection.
RUN_FOLDER = None

if RUN_FOLDER is None:
    candidates = sorted([p for p in DATASET_ROOT.glob("ascon_tighttrigger_30k_per_byte_*") if p.is_dir()])
    if not candidates:
        raise FileNotFoundError(f"No ASCON dataset run folders found under {DATASET_ROOT}")
    RUN_FOLDER = candidates[-1]

print("Checking:", RUN_FOLDER)


In [ ]:
required_meta_cols = [
    "algorithm", "target_byte", "trace_index", "attempt_index", "key_hex", "nonce_hex",
    "key_byte_value", "key_byte_hex", "key_byte_hw", "output_hex", "trigger_count", "capture_ret",
] + [f"bit{i}_lsb" for i in range(8)]

rows = []
for b in range(16):
    byte_dir = RUN_FOLDER / f"byte_{b:02d}"
    trace_files = sorted(byte_dir.glob(f"byte{b:02d}_chunk_*_traces.npz"))
    meta_files = sorted(byte_dir.glob(f"byte{b:02d}_chunk_*_metadata.csv"))

    total_traces = 0
    total_meta = 0
    shape_ok = True
    missing_cols = []
    bad_pairs = 0

    for tf, mf in zip(trace_files, meta_files):
        with np.load(tf) as z:
            key = "traces" if "traces" in z.files else z.files[0]
            X = z[key]
        meta = pd.read_csv(mf)
        total_traces += len(X)
        total_meta += len(meta)
        if len(X.shape) != 2:
            shape_ok = False
        if len(X) != len(meta):
            bad_pairs += 1
        for c in required_meta_cols:
            if c not in meta.columns and c not in missing_cols:
                missing_cols.append(c)

    rows.append({
        "byte": b,
        "trace_chunks": len(trace_files),
        "metadata_chunks": len(meta_files),
        "total_traces": total_traces,
        "total_metadata_rows": total_meta,
        "shape_ok": shape_ok,
        "bad_pairs": bad_pairs,
        "missing_cols": ";".join(missing_cols),
        "ok": len(trace_files) == len(meta_files) and total_traces == total_meta and shape_ok and bad_pairs == 0 and len(missing_cols) == 0,
    })

summary = pd.DataFrame(rows)
out = RUN_FOLDER / "dataset_verification_summary.csv"
summary.to_csv(out, index=False)
print("Saved:", out)
summary
